In [ ]:
# E-Commerce Sales & Customer Behaviour Analysis
**Dataset:** Olist Brazilian E-Commerce | 100,000+ real orders  
**Tools:** Python · SQL · Power BI  

---
## Business Story
We analyse real e-commerce data to answer 3 questions:
1. **What does our sales performance look like?** → EDA
2. **Can we predict if a customer will be unhappy?** → Machine Learning
3. **Who are our most valuable customers?** → RFM Segmentation
4. **Can SQL confirm our findings?** → SQL Analysis

In [ ]:

# Phase 1 — Data Loading
#Load all 9 Olist CSV files and merge them into one master table.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import sqlite3
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
print('Libraries loaded!')

In [ ]:
# Load all 9 CSV files
# Make sure all files are in your project folder (same folder as this notebook)

orders       = pd.read_csv('olist_orders_dataset.csv')
customers    = pd.read_csv('olist_customers_dataset.csv')
order_items  = pd.read_csv('olist_order_items_dataset.csv')
products     = pd.read_csv('olist_products_dataset.csv')
sellers      = pd.read_csv('olist_sellers_dataset.csv')
payments     = pd.read_csv('olist_order_payments_dataset.csv')
reviews      = pd.read_csv('olist_order_reviews_dataset.csv')
category_eng = pd.read_csv('product_category_name_translation.csv')

print(f'orders       : {orders.shape}')
print(f'customers    : {customers.shape}')
print(f'order_items  : {order_items.shape}')
print(f'products     : {products.shape}')
print(f'payments     : {payments.shape}')
print(f'reviews      : {reviews.shape}')
print('All files loaded!')

In [ ]:
# Translate product categories from Portuguese to English
products = products.merge(category_eng, on='product_category_name', how='left')
products['category'] = products['product_category_name_english'].fillna(products['product_category_name'])

# Aggregate payments — one row per order
pay = payments.groupby('order_id').agg(
    total_payment_value  = ('payment_value', 'sum'),
    payment_installments = ('payment_installments', 'max'),
    payment_type         = ('payment_type', lambda x: x.mode()[0])
).reset_index()

# Aggregate order items — one row per order
items = order_items.groupby('order_id').agg(
    item_count    = ('order_item_id', 'count'),
    total_price   = ('price', 'sum'),
    total_freight = ('freight_value', 'sum'),
    product_id    = ('product_id', 'first'),
    seller_id     = ('seller_id', 'first')
).reset_index()

# Aggregate reviews — one row per order
rev = reviews.groupby('order_id').agg(review_score=('review_score','mean')).reset_index()

# Merge everything into one master table
df = orders.copy()
df = df.merge(customers,  on='customer_id',  how='left')
df = df.merge(items,      on='order_id',     how='left')
df = df.merge(products[['product_id','category']], on='product_id', how='left')
df = df.merge(sellers[['seller_id','seller_state']], on='seller_id', how='left')
df = df.merge(pay,        on='order_id',     how='left')
df = df.merge(rev,        on='order_id',     how='left')

print(f'Master table shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print('Phase 1 complete!')

In [ ]:

# Phase 2 — Data Cleaning
#Fix missing values, convert dates, engineer new columns, keep only delivered orders.

In [ ]:
# Step 1 — Convert date columns from text to datetime
date_cols = ['order_purchase_timestamp','order_approved_at',
             'order_delivered_carrier_date','order_delivered_customer_date',
             'order_estimated_delivery_date']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Step 2 — Fix missing values
df['order_approved_at'] = df['order_approved_at'].fillna(df['order_purchase_timestamp'])
df['category']          = df['category'].fillna('Unknown')
df['review_score']      = df['review_score'].fillna(df['review_score'].median())
for col in ['product_weight_g','product_length_cm','product_height_cm','product_width_cm']:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

# Step 3 — Keep only delivered orders (standard for sales analysis)
df = df[df['order_status'] == 'delivered'].copy()
df = df.dropna(subset=['total_payment_value'])

# Step 4 — Create new useful columns
df['delivery_days']      = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days
df['days_to_estimated']  = (df['order_estimated_delivery_date'] - df['order_delivered_customer_date']).dt.days
df['is_late']            = (df['days_to_estimated'] < 0).astype(int)
df['order_month']        = df['order_purchase_timestamp'].dt.month
df['order_year']         = df['order_purchase_timestamp'].dt.year
df['order_month_yr']     = df['order_purchase_timestamp'].dt.to_period('M').astype(str)
df['order_dayofweek']    = df['order_purchase_timestamp'].dt.day_name()
df['order_value']        = df['total_price'] + df['total_freight']

# Step 5 — Remove outliers
df = df[(df['total_price'] > 0) & (df['delivery_days'] >= 0) & (df['delivery_days'] <= 120)]

# Save
df.to_csv('master_cleaned.csv', index=False)
print(f'Clean data: {df.shape[0]:,} rows x {df.shape[1]} columns')
print('Saved → master_cleaned.csv')
print('Phase 2 complete!')

In [ ]:

# Phase 3 — Exploratory Data Analysis (EDA)
### Business Question: What does our sales performance look like?
#We use charts to find trends, top products, best locations, and delivery issues.

In [ ]:
# Chart 1 — Monthly Revenue Trend
# Is our business growing month by month?

monthly = df.groupby('order_month_yr')['total_payment_value'].sum().reset_index()
monthly.columns = ['Month','Revenue']
monthly = monthly.sort_values('Month')

plt.figure(figsize=(14, 4))
plt.plot(monthly['Month'], monthly['Revenue'], marker='o', color='steelblue', linewidth=2)
plt.fill_between(monthly['Month'], monthly['Revenue'], alpha=0.1, color='steelblue')
plt.title('Monthly Revenue — Is Our Business Growing?', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.gca().yaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f'R${x/1e6:.1f}M'))
plt.tight_layout()
plt.savefig('chart1_revenue.png', bbox_inches='tight')
plt.show()

peak = monthly.loc[monthly['Revenue'].idxmax()]
print(f'Peak month   : {peak["Month"]} — R${peak["Revenue"]:,.0f}')
print(f'Total revenue: R${monthly["Revenue"].sum():,.0f}')

In [ ]:
# Chart 2 — Top Categories and Top States
# Where is the money coming from?

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Where Is the Money Coming From?', fontsize=14, fontweight='bold')

top_cat = df.groupby('category')['total_payment_value'].sum().sort_values().tail(8)
axes[0].barh(top_cat.index, top_cat.values, color='steelblue', edgecolor='white')
axes[0].set_title('Top 8 Categories by Revenue')
axes[0].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f'R${x/1e6:.1f}M'))

top_states = df.groupby('customer_state')['total_payment_value'].sum().sort_values().tail(8)
axes[1].barh(top_states.index, top_states.values, color='mediumseagreen', edgecolor='white')
axes[1].set_title('Top 8 States by Revenue')
axes[1].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f'R${x/1e6:.1f}M'))

plt.tight_layout()
plt.savefig('chart2_categories_states.png', bbox_inches='tight')
plt.show()

print(f'Top category : {top_cat.index[-1]}')
print(f'Top state    : {top_states.index[-1]}')

In [ ]:
# Chart 3 — Delivery Performance & Impact on Reviews
# Do late deliveries make customers unhappy?

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Delivery Performance — Does It Affect Customer Happiness?', fontsize=13, fontweight='bold')

# Delivery days histogram
axes[0].hist(df['delivery_days'].dropna(), bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(df['delivery_days'].median(), color='red', linestyle='--',
                label=f'Median: {df["delivery_days"].median():.0f} days')
axes[0].set_title('How Long Does Delivery Take?')
axes[0].set_xlabel('Days')
axes[0].legend()

# On-time vs late pie chart
late = df['is_late'].value_counts().sort_index()
axes[1].pie(late.values, labels=['On Time','Late'],
            autopct='%1.1f%%', colors=['mediumseagreen','tomato'], startangle=90)
axes[1].set_title('On-Time vs Late Deliveries')

# Review score comparison
rev_late = df.groupby('is_late')['review_score'].mean().round(2).sort_index()
rev_late.index = ['On Time','Late']
bars = axes[2].bar(rev_late.index, rev_late.values,
                   color=['mediumseagreen','tomato'], edgecolor='white', width=0.4)
for bar, val in zip(bars, rev_late.values):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()-0.3,
                 str(val), ha='center', fontsize=13, fontweight='bold', color='white')
axes[2].set_title('Avg Review: On Time vs Late')
axes[2].set_ylim(0, 5.5)

plt.tight_layout()
plt.savefig('chart3_delivery.png', bbox_inches='tight')
plt.show()

diff = rev_late['On Time'] - rev_late['Late']
print(f'Late delivery rate : {df["is_late"].mean()*100:.1f}%')
print(f'Review score drop  : {diff:.2f} points lower when late')
print(f'Insight: Faster delivery = happier customers!')

In [ ]:
# Chart 4 — When Do Customers Buy?
# Best days and months to run promotions

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('When Do Customers Buy?', fontsize=14, fontweight='bold')

day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
by_day = df['order_dayofweek'].value_counts().reindex(day_order)
axes[0].bar(by_day.index, by_day.values, color='steelblue', edgecolor='white')
axes[0].set_title('Orders by Day of Week')
axes[0].tick_params(axis='x', rotation=30)

month_map = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
             7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
by_month = df['order_month'].value_counts().sort_index()
by_month.index = [month_map[m] for m in by_month.index]
axes[1].bar(by_month.index, by_month.values, color='mediumseagreen', edgecolor='white')
axes[1].set_title('Orders by Month')

plt.tight_layout()
plt.savefig('chart4_timing.png', bbox_inches='tight')
plt.show()

print(f'Busiest day   : {by_day.idxmax()}')
print(f'Busiest month : {by_month.idxmax()}')
print(f'Tip: Run promotions on {by_day.idxmax()}s and in {by_month.idxmax()} for maximum reach!')

In [ ]:
# EDA Key Numbers — copy these into your resume!

print('=' * 55)
print('EDA SUMMARY — KEY BUSINESS NUMBERS')
print('=' * 55)
print(f'  Total Revenue        : R${df["total_payment_value"].sum():,.0f}')
print(f'  Total Orders         : {df["order_id"].nunique():,}')
print(f'  Avg Order Value      : R${df["total_payment_value"].mean():,.2f}')
print(f'  Top Category         : {df.groupby("category")["total_payment_value"].sum().idxmax()}')
print(f'  Top State            : {df["customer_state"].value_counts().idxmax()}')
print(f'  Median Delivery Time : {df["delivery_days"].median():.0f} days')
print(f'  Late Delivery Rate   : {df["is_late"].mean()*100:.1f}%')
print(f'  On-Time Review Score : {df[df["is_late"]==0]["review_score"].mean():.2f} / 5')
print(f'  Late Review Score    : {df[df["is_late"]==1]["review_score"].mean():.2f} / 5')
print('=' * 55)
print('Phase 3 complete!')

In [ ]:

# Phase 4 — Machine Learning
### Business Question: Can we predict if a customer will leave a bad review?

'''If we predict a bad review **before** it happens, we can contact the customer and fix the issue — saving the relationship.

| Model | What it does |
|---|---|
| Logistic Regression | Draws a line to separate happy vs unhappy customers |
| Decision Tree | A Yes/No flowchart — easy for anyone to understand |
| Random Forest | 100 decision trees vote together — usually most accurate |'''

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

# Target: 0 = LOW review (1-2 stars), 1 = HIGH review (4-5 stars)
# Remove neutral score 3 — we want clear low vs high
ml = df[df['review_score'] != 3].copy()
ml['target'] = (ml['review_score'] >= 4).astype(int)

features = ['delivery_days','is_late','days_to_estimated',
            'total_price','total_freight','item_count',
            'payment_installments','order_month']

ml = ml[features + ['target']].dropna()
X, y = ml[features], ml['target']

# 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

sc = StandardScaler()
Xs_train = sc.fit_transform(X_train)
Xs_test  = sc.transform(X_test)

print(f'Training rows : {len(X_train):,}')
print(f'Testing rows  : {len(X_test):,}')
print(f'LOW reviews   : {(y==0).sum():,}  |  HIGH reviews: {(y==1).sum():,}')

In [ ]:
# Train all 3 models

lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(Xs_train, y_train)
lr_acc = accuracy_score(y_test, lr.predict(Xs_test))

dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
dt_acc = accuracy_score(y_test, dt.predict(X_test))

rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_acc  = accuracy_score(y_test, rf_pred)

# Chart: model comparison + feature importance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Machine Learning Results', fontsize=14, fontweight='bold')

bars = axes[0].bar(['Logistic\nRegression','Decision\nTree','Random\nForest'],
                   [lr_acc*100, dt_acc*100, rf_acc*100],
                   color=['#3498db','#f39c12','#2ecc71'], edgecolor='white', width=0.5)
for bar, val in zip(bars, [lr_acc*100, dt_acc*100, rf_acc*100]):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()-4,
                 f'{val:.1f}%', ha='center', fontsize=12, fontweight='bold', color='white')
axes[0].set_title('Which Model is Most Accurate?')
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_ylim(0, 100)

imp = pd.Series(rf.feature_importances_, index=features).sort_values()
axes[1].barh(imp.index, imp.values, color='steelblue', edgecolor='white')
axes[1].set_title('What Drives Customer Satisfaction?')
axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.savefig('chart5_ml_results.png', bbox_inches='tight')
plt.show()

print(f'Logistic Regression : {lr_acc*100:.1f}%')
print(f'Decision Tree       : {dt_acc*100:.1f}%')
print(f'Random Forest       : {rf_acc*100:.1f}%  <- Best')
print(f'Top factor          : {imp.idxmax()} — this is the #1 driver of customer satisfaction')

In [ ]:
# Decision Tree chart — anyone can read this like a flowchart

plt.figure(figsize=(18, 6))
plot_tree(dt, max_depth=3, feature_names=features,
          class_names=['LOW','HIGH'], filled=True, rounded=True, fontsize=9)
plt.title('Decision Tree — How the Model Predicts Reviews (Read top to bottom)',
          fontsize=13, fontweight='bold')
plt.savefig('chart6_decision_tree.png', bbox_inches='tight')
plt.show()
print('Each box = a decision. Follow Yes/No branches from top to reach LOW or HIGH prediction.')

In [ ]:
# Confusion Matrix — how many predictions were correct?

cm = confusion_matrix(y_test, rf_pred)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=['LOW Review','HIGH Review']).plot(
    ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Random Forest — {rf_acc*100:.1f}% Accurate', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('chart7_confusion_matrix.png', bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'Correctly predicted unhappy customers : {tn:,}')
print(f'Correctly predicted happy customers   : {tp:,}')
print('Phase 4 complete!')

In [ ]:

# Phase 5 — RFM Customer Segmentation
### Business Question: Who are our most valuable customers?

'''Not all customers are equal. RFM scores every customer on:
- **R (Recency)** — How recently did they buy? *(recent = more likely to buy again)*
- **F (Frequency)** — How often do they buy? *(more = more loyal)*
- **M (Monetary)** — How much do they spend? *(more = more valuable)*

This puts customers into 4 groups: **Champions | Loyal | At Risk | Lost**'''

In [ ]:
# Calculate RFM scores for every customer

ref_date = df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

rfm = df.groupby('customer_unique_id').agg(
    recency   = ('order_purchase_timestamp', lambda x: (ref_date - x.max()).days),
    frequency = ('order_id', 'count'),
    monetary  = ('total_payment_value', 'sum')
).reset_index()

# Score 1-4 (4 = best)
rfm['R'] = pd.qcut(rfm['recency'],  q=4, labels=[4,3,2,1]).astype(int)
rfm['F'] = pd.qcut(rfm['frequency'].rank(method='first'), q=4, labels=[1,2,3,4]).astype(int)
rfm['M'] = pd.qcut(rfm['monetary'], q=4, labels=[1,2,3,4]).astype(int)

# Assign segment based on scores
def segment(row):
    if row['R'] >= 3 and row['F'] >= 3 and row['M'] >= 3: return 'Champions'
    elif row['R'] >= 3 and row['F'] >= 2:                  return 'Loyal'
    elif row['R'] == 2:                                     return 'At Risk'
    else:                                                   return 'Lost'

rfm['Segment'] = rfm.apply(segment, axis=1)
rfm.to_csv('rfm_segments.csv', index=False)

print(f'Total customers: {len(rfm):,}')
print(rfm['Segment'].value_counts().to_string())
print('Saved → rfm_segments.csv')

In [ ]:
# Visualise segments — who are they and how much do they contribute?

summary = rfm.groupby('Segment').agg(
    Customers = ('customer_unique_id','count'),
    Revenue   = ('monetary','sum'),
    Avg_Spend = ('monetary','mean')
).reset_index()

summary['Revenue_%']  = (summary['Revenue']  / summary['Revenue'].sum()  * 100).round(1)
summary['Customer_%'] = (summary['Customers'] / summary['Customers'].sum() * 100).round(1)

seg_order  = ['Champions','Loyal','At Risk','Lost']
seg_colors = ['#2ecc71','#3498db','#f39c12','#e74c3c']
summary    = summary.set_index('Segment').reindex(seg_order).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Who Are Our Customers — and Who Drives Our Revenue?', fontsize=13, fontweight='bold')

axes[0].bar(summary['Segment'], summary['Customers'], color=seg_colors, edgecolor='white')
axes[0].set_title('Customers per Segment')
for i, v in enumerate(summary['Customers']):
    axes[0].text(i, v+100, f'{v:,}', ha='center', fontsize=9, fontweight='bold')

axes[1].bar(summary['Segment'], summary['Revenue'], color=seg_colors, edgecolor='white')
axes[1].set_title('Revenue per Segment')
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f'R${x/1e6:.1f}M'))
for i, v in enumerate(summary['Revenue']):
    axes[1].text(i, v+20000, f'R${v/1e6:.1f}M', ha='center', fontsize=9, fontweight='bold')

axes[2].pie(summary['Revenue_%'], labels=summary['Segment'],
            autopct='%1.1f%%', colors=seg_colors, startangle=90)
axes[2].set_title('Revenue Share by Segment')

plt.tight_layout()
plt.savefig('chart8_rfm.png', bbox_inches='tight')
plt.show()

champ = summary[summary['Segment']=='Champions'].iloc[0]
print(f'Champions: {champ["Customer_%"]}% of customers → {champ["Revenue_%"]}% of revenue')
print('Small loyal group drives most of the business!')

In [ ]:
# Business Recommendations — what should the company do?

actions = {
    'Champions': 'Reward them — VIP discounts, early access, loyalty points.',
    'Loyal'    : 'Keep them engaged — personalised offers, ask for referrals.',
    'At Risk'  : 'Re-engage fast — send a limited-time we miss you coupon.',
    'Lost'     : 'Win-back campaign — big discount. If no response, reduce spend.'
}

print('=' * 60)
print('BUSINESS RECOMMENDATIONS BY CUSTOMER SEGMENT')
print('=' * 60)
for seg in seg_order:
    row = summary[summary['Segment']==seg].iloc[0]
    print(f'\n  {seg}')
    print(f'  {row["Customers"]:,} customers | {row["Revenue_%"]}% of revenue | Avg spend R${row["Avg_Spend"]:,.0f}')
    print(f'  Action: {actions[seg]}')
print('\nPhase 5 complete!')